# ASF — quickstart (anonymous search)

Run a small Sentinel-1 SLC catalog search over a bounded bbox / time window. **No credentials required** — `asf_search.geo_search` (and everything `backend._search()` invokes for the search mode) is anonymous. EDL credentials are only needed for the download step.

Needs the `[asf]` extra (`pip install earthlens[asf]`).

## Setup

Build the facade in search mode. The bbox is a small Iceland-rift-zone window where Sentinel-1 ascends every few days, so a short time window normally yields a handful of results.

In [ ]:
from earthlens.earthlens import EarthLens

el = EarthLens(
    data_source='asf',
    variables=['sentinel-1-slc'],
    start='2024-06-01',
    end='2024-06-15',
    lat_lim=[63.8, 64.4],
    lon_lim=[-21.7, -21.0],
    path='asf_out',
    flight_direction='ASCENDING',
    max_results=10,
)
el.datasource._mode

## Dry-run inspection

`backend._search()` returns one `RemoteProduct` per matching `ASFProduct`. We can inspect it without downloading anything — useful for sizing a request before paying for the EDL login + bulk fetch.

In [ ]:
products = el.datasource._search()
print(f'{len(products)} product(s) match the request')
for p in products[:5]:
    print(f'  {p.id:62s} {p.metadata["fileName"]}')

### How big is the request?

Each `ASFProduct` carries a `bytes` property — sum it to get the on-disk weight.

In [ ]:
total = sum(p.metadata['product'].properties.get('bytes', 0) for p in products)
print(f'Total download size: {total / 1e9:.2f} GB across {len(products)} product(s)')

### What's in `metadata`?

Inspecting the first product shows the metadata shape the backend records. In search mode the baseline keys (`perpendicularBaseline`, `temporalBaseline`) are intentionally omitted — they only exist on stacked products. See the [InSAR stack notebook](sentinel1_insar_stack.ipynb).

In [ ]:
first = products[0]
for key, value in first.metadata.items():
    print(f'  {key:24s}: {value if key != "product" else "<ASFProduct>"}')

## Downloading (gated on EDL credentials)

When you have `EARTHDATA_TOKEN` (or `EARTHDATA_USERNAME` + `EARTHDATA_PASSWORD`) set, the same backend can fetch:

```python
paths = el.download()   # gated by EDL creds
```

The call returns a `list[Path]` of the downloaded SLC archives. Idempotent re-runs skip files already on disk. Credentials and the error path are documented under [Authentication](../../reference/asf/authentication.md).